# 机器人运动控制 (Locomotion) 
## 传统MCP方法

### MPC 优化范式：
$$ \text{minimize}_{\mathbf{u}(\cdot)} \int_{t_s}^{t_f} l(\mathbf{x}(t), \mathbf{u}(t), t) dt $$
需满足初始状态、动力学方程 $\dot{\mathbf{x}} = f(\mathbf{x}, \mathbf{u}, t)$ 及等式/不等式约束。

**一句话解释：** 这个积分是在计算**从现在开始，到未来一段时间内，机器人为了完成任务所付出的“总代价”（Total Cost）。**

我们的目标是 $\text{minimize}$（最小化）这个总代价。也就是说，我们想找到一种控制方法，让机器人既能完成任务，又最省力、最平稳。

**形象类比：打车软件的路线规划**
假设你要从家（$t_s$）去机场（$t_f$）。
*   **$\mathbf{x}(t)$（状态）：** 你的位置、车速。
*   **$\mathbf{u}(t)$（控制）：** 踩油门的力度、打方向盘的角度。
*   **$l(\dots)$（瞬时代价）：** 你每一秒钟心里的“不爽程度”。

**为什么是积分？**
因为你的旅程是一个连续的过程。
*   如果在第 1 分钟，司机急刹车，你不爽（$l$ 很大）。
*   如果在第 10 分钟，司机为了绕路多烧了油，钱多了，你不爽（$l$ 变大）。
*   如果在第 30 分钟，司机开得太慢，让你快迟到了，你也不爽（$l$ 变大）。

**积分 $\int$ 的作用，就是把你这一路上每一秒钟的“不爽程度”全部加起来。** MPC 的任务就是算出怎么踩油门、怎么打方向，才能让你这一路上的“总不爽值”最低。

#### 公式符号详解

我们回到机器人（比如一只 Unitree Go2 机器狗）的语境下，看看这些符号代表什么。

1. $\mathbf{x}(t)$：状态向量 (State Vector) —— "我现在怎么样？"
*   **定义：** 描述机器人在 $t$ 时刻的物理状态。
*   **对于机器狗：** 它通常包含 **位置**（$p_x, p_y, p_z$）、**速度**（$v_x, v_y, v_z$）、**身体朝向**（Roll, Pitch, Yaw）以及**角速度**。
*   **意义：** 告诉控制器机器人现在的身体姿态和运动趋势。

2. $\mathbf{u}(t)$：控制输入 (Control Input) —— "我打算怎么做？"
*   **定义：** 我们可以直接控制的变量。
*   **对于机器狗：** 通常是 12 个电机的**力矩**（Torque）或者**目标关节角度**。
*   **意义：** 这是优化问题的“解”。我们求的就是这一串 $u(t)$，告诉电机下一毫秒该出多少力。

3. $l(\mathbf{x}, \mathbf{u}, t)$：瞬时代价函数 (Running Cost / Stage Cost) —— "这一刻我做得好不好？"
这是最核心的部分！它定义了什么是“好”，什么是“坏”。在 MPC 中，这个函数通常被设计成两部分的加和：

$$ l(\mathbf{x}, \mathbf{u}) = \underbrace{||\mathbf{x} - \mathbf{x}_{ref}||^2_Q}_{\text{任务误差}} + \underbrace{||\mathbf{u}||^2_R}_{\text{能量消耗}} $$

*   **第一部分（任务误差）：**
    *   意思：我现在的状态 $\mathbf{x}$ 和目标状态 $\mathbf{x}_{ref}$ 差得远不远？
    *   例子：目标是让狗以 1m/s 前进。如果现在速度只有 0.2m/s，这个误差就很大，代价 $l$ 就很高。**最小化它，就是逼着机器人去追赶目标。**
*   **第二部分（能量消耗）：**
    *   意思：我为了达到目标，花了多大的力气 $\mathbf{u}$？
    *   例子：为了达到 1m/s，我可以慢慢加速（$u$ 小），也可以一脚地板油（$u$ 极大）。地板油虽然快，但费电且可能打滑。**最小化它，就是逼着机器人动作要“温柔”、省电、高效。**

4. $t_s$ 和 $t_f$：积分区间 —— "我看多远？"
*   **$t_s$ (Start Time)：** 当前时刻。
*   **$t_f$ (Final Time)：** 预测视野的终点。
*   **MPC 的特点（Receding Horizon）：**
    MPC 不是算完一辈子的账，它只算**未来一小段时间**（比如未来 0.5 秒）。
    *   $t_f - t_s$ 叫做**预测时域（Horizon）**。
    *   如果你看太短（Horizon 太小），机器人会“鼠目寸光”，只顾眼前不顾后面（比如为了快点到，结果冲太猛停不下来撞墙）。
    *   如果你看太长（Horizon 太大），计算量会爆炸，计算机算不过来。

5. $\text{minimize}_{\mathbf{u}(\cdot)}$：优化目标
*   这就好比在问数学老师：“请帮我找到一条曲线 $\mathbf{u}(t)$，使得上面算出来的那个积分值（总代价）最小。”

#### 为什么要最小化这个积分？

如果不最小化这个积分，机器人会出现什么奇葩行为？

1.  **如果只看终点（没有积分过程）：**
    机器人可能会为了到达终点，中间做出一系列疯狂的动作（比如翻滚着过去，或者把腿扭断了过去），只要最后时刻到了就行。**积分约束了整个过程必须是平滑、合理的。**

2.  **如果不考虑控制量 $\mathbf{u}$ 的代价：**
    机器人会一直使用最大力矩工作，哪怕只需要走慢一点点。这会导致电机过热、电池瞬间耗干，或者动作极其僵硬、震荡。**积分项里的 $\mathbf{u}$ 也是在保护硬件。**

#### 对比：MPC vs. RL

| 维度 | 模型预测控制 (MPC) | 强化学习 (RL) |
| :--- | :--- | :--- |
| **模型依赖** | 强依赖 (System Dynamics, Constraints) | **Model-Free** (不依赖显式动力学模型) |
| **运行机制** | 在线求解优化问题 (Real-time Optimization) | 离线训练策略，在线推理 (Inference) |
| **约束处理** | 自然处理硬约束 | 将约束转化为 **Reward** (软约束) |
| **适应性** | 适合已知、稳定环境 | 适应复杂、未知、非结构化环境 |
| **鲁棒性** | 较差 (受限于模型精度) | **强** (Sim-to-Real 表现更好) |

## RL 在 Locomotion 中的 MDP 构建

### 1. 状态空间 (State Space)
通常包含本体感知 (Proprioception) 和 外部感知 (Exteroception)：
*   **本体感知：**
    *   IMU数据：角速度 $\omega$、加速度 $a$。
    *   关节数据：关节位置 $q$、关节速度 $\dot{q}$。
    *   历史动作：$a_{t-1}$。
*   **表示方式：**
    *   **Joint Space (关节空间)：** $[q_1, \dot{q}_1, ..., q_n, \dot{q}_n]$，低维，Locomotion 任务常用。
    *   Link Space (连杆空间)：编码空间关系，常用于跳舞或人机交互任务。
*   **外部感知：** 深度相机、激光雷达点云（用于避障、地形适应）。

### 2. 动作空间 (Action Space)
*   **PD 控制器架构：** RL 输出通常不是直接力矩，而是 **PD控制器的残差目标位置**。
    $$ \tau = K_p (q_{ref} + a_{RL} - q) + K_d (\dot{q}_{ref} - \dot{q}) $$
    *   相比 Torque Control，PD Control 可以在低频策略下输出高频力矩，且 Sim-to-Real 效果更好。
    *   一般不使用积分项 (I)，因为非稳态运动会导致误差累积震荡。

### 3. 奖励函数设计 (Reward Engineering)
目前缺乏通用规则，通常包含两类：
*   **任务奖励 (Task Rewards)：** 鼓励完成目标。
    *   线速度/角速度追踪：$\exp(-\|v_{target} - v_{measure}\|^2)$
    *   姿态保持：保持躯干高度、水平。
*   **正则化奖励 (Regularization Rewards)：** 抑制不自然或危险行为。
    *   能量惩罚（Energy consumption）。
    *   动作平滑度（Action smoothness/rate）。
    *   脚部撞击力、关节限位惩罚。

### 4. 仿真环境
*   **主流选择：** Isaac Gym (NVIDIA, GPU并行加速), MuJoCo (物理引擎准确)。

# 夹爪操作 (Gripper Manipulation)
该部分主要探讨如何使用平行夹爪或吸盘进行6自由度（6DoF）的抓取任务，重点在于如何利用强化学习（RL）和模仿学习（IL）解决高维空间动作规划问题。

## 1.1 基于分阶段模仿与RL的策略 (On-policy)
*   **核心思想**：将复杂的抓取任务分解为连续的子阶段，降低奖励设计难度。
*   **阶段划分**：
    1.  Orienting (调整末端姿态)
    2.  Approaching (接近目标)
    3.  Closing (闭合夹爪)
*   **算法架构**：
    *   **模仿学习**：GAIL (Generative Adversarial Imitation Learning)
    *   **强化学习**：PPO (Proximal Policy Optimization)
    *   **网络结构**：使用LSTM处理序列信息。

### GAIL (Generative Adversarial Imitation Learning) 
#### 直观类比：从 GAN 到 GAIL
**(1) GAN (生成图片)**
*   **任务**：造假币。
*   **Generator (生成器)**：印假币的人。目标是印出连验钞机都认不出来的假币。
*   **Discriminator (判别器)**：验钞机。目标是精准分辨出哪张是真币（训练集数据），哪张是假币（生成器造的）。
*   **结局**：当验钞机无法分辨真假时，假币就做得和真币一样了。

**(2) GAIL (生成动作策略)**
*   **任务**：模仿迈克尔·杰克逊跳舞。
*   **Actor (演员/策略)**：模仿者。目标是跳出一段舞，让评委觉得这是迈克尔·杰克逊本人跳的。
*   **Discriminator (判别器)**：毒舌评委。目标是分辨出眼前这个动作是迈克尔·杰克逊做的（专家数据），还是模仿者做的（Actor 产生的）。
*   **结局**：当评委分不清谁是真身时，模仿者就学会了完美的舞步。

| 特性 | GAN (生成对抗网络) | GAIL (生成对抗模仿学习) |
| :--- | :--- | :--- |
| **生成模块** | **Generator ($G$)** <br> 输入噪声 $z$，直接输出样本（如图片）。 | **Actor ($\pi$)** <br> 输入状态 $s$，输出动作 $a$，**与环境交互**产生轨迹。 |
| **判别模块** | **Discriminator ($D$)** <br> 输入图片 $x$。 | **Discriminator ($D$)** <br> 输入“状态-动作对” $(s, a)$。 |
| **判别器的作用** | **二分类器**：<br>输出概率值，判断输入图片是“真图”还是“假图”。 | **奖励函数提供者 (Reward Giver)**：<br>判断 $(s, a)$ 是“专家产生的”还是“Actor产生的”。<br>**关键点：判别器的输出被转化为 Reward 喂给 RL 算法。** |
| **优化方法** | 反向传播 (Backpropagation) <br> 梯度直接流经 $D$ 传给 $G$。 | **强化学习 (PPO/TRPO)** <br> 因为环境不可导，不能直接反向传播，必须用 RL 更新 Actor。 |

####  Discriminator 的作用
在 GAIL 中，Discriminator 的目标是最小化以下分类误差：
*   给 **专家数据 $(s_E, a_E)$** 打高分（趋近 1）。
*   给 **Actor 数据 $(s_\pi, a_\pi)$** 打低分（趋近 0）。

对于 **Actor** 来说，它的目标是“骗过”Discriminator。因此，GAIL 定义了一个由 Discriminator 导出的**奖励函数 (Surrogate Reward)**：
$$ r(s, a) = - \log(1 - D(s, a)) $$
或者简单理解为：$D(s, a)$ 越接近 1（越像专家），Reward 越高。

#### GAIL的运作模式

在 GAIL 中，其实包含了两套系统在同时运作：

1.  **对抗系统 (Adversarial System)**：
    *   **角色**：Actor (策略) vs. Discriminator (判别器)。
    *   **目的**：**学习“什么是好的动作”**。Discriminator 通过不断学习分辨专家和菜鸟，告诉 Actor 现在的动作像不像专家。

2.  **强化学习系统 (RL System - 比如 PPO)**：
    *   **角色**：Actor (策略) + **Critic (价值函数)**。
    *   **目的**：**“如何执行好的动作”**。
    *   **Critic 的作用**：这里的 Critic **不是**用来分辨真假的，而是 PPO/TRPO 算法自带的 Value Function。它用来估计处于当前状态 $s$ 能够获得的长期回报（Value）。它帮助 Actor 减少方差，更稳定地更新参数。

**总结结构图：**

$$
\text{Expert Data} \xrightarrow{\text{正样本}} \fbox{Discriminator (D)} \xleftarrow{\text{负样本}} \text{Actor产生的轨迹}
$$
$$
\downarrow
$$
$$
\text{D 输出 Reward}
$$
$$
\downarrow
$$
$$
\text{RL Update (PPO)}: \begin{cases} \text{Actor}(\pi): \text{最大化 D 给的 Reward} \\ \text{Critic}(V): \text{估计 Reward 的累积期望 (为了让 Actor 训练更好)} \end{cases}
$$

#### 算法设计思路详解

##### **GAN 的数学目标**
是一个最小最大博弈 (Minimax Game)：
$$ \min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_{z}}[\log(1 - D(G(z)))] $$
*   $D$ 想最大化分辨能力。
*   $G$ 想最小化 $D$ 的分辨成功率。

##### **GAIL 的数学目标**
GAIL 证明了模仿学习本质上是在匹配 **占用度量 (Occupancy Measure)**，即专家访问 $(s, a)$ 的频率分布 $\rho_E$ 和 Actor 访问 $(s, a)$ 的频率分布 $\rho_\pi$。

GAIL 的目标是最小化这两个分布的 Jensen-Shannon 散度：
$$ \min_\pi \max_D \mathbb{E}_{\pi_E} [\log D(s, a)] + \mathbb{E}_{\pi} [\log(1 - D(s, a))] - \lambda H(\pi) $$

**算法流程步骤：**

1.  **采样 (Rollout)**：Actor $\pi_\theta$ 在环境中跑动，收集一批轨迹数据 $(s, a)$。
2.  **更新 Discriminator**：
    *   拿出一批专家数据 $(s_E, a_E)$ 标记为 1。
    *   拿出刚才 Actor 跑的数据 $(s_\pi, a_\pi)$ 标记为 0。
    *   训练 $D$ 分辨这两拨数据。
3.  **计算奖励**：
    *   把 Actor 的数据输入更新后的 $D$。
    *   计算每个人为奖励 $r = -\log(1 - D(s, a))$（或者类似形式）。
4.  **更新 Actor (和 RL Critic)**：
    *   把计算出的 $r$ 当作环境给的 Reward。
    *   使用 **PPO** 或 **TRPO** 算法更新 Actor 的参数 $\theta$（此时会用到 RL 里的 Critic 来计算优势函数 Advantage）。
5.  **循环**：回到步骤 1，直到 Actor 的行为让 Discriminator 无法分辨（即 $D$ 输出恒为 0.5）。


## 1.2 GA-DDPG: 目标辅助的行动者-评论家算法 (Off-policy)
DDPG (Deep Deterministic Policy Gradient)针对6D抓取中的闭环控制问题，特别是接触丰富的场景。
*   **核心算法**：DDPG (Deep Deterministic Policy Gradient)
    *   **Actor** $\pi_\theta(s)$: 学习策略。
    *   **Critic** $Q_\phi(s, a)$: 近似Q函数。
    *   **优化目标**：最大化Q值 $\max_\theta \mathbb{E}_{s \sim D} [Q_\phi(s, \pi_\theta(s))]$
*   **状态与动作**：
    *   State $s_t$: 物体的3D点云。
    *   Action $a_t$: 末端执行器的3D位移 + 3D旋转。
*   **训练数据构成 (Replay Buffer)**：
    1.  **$D_{expert}$ (专家数据)**：来自OMG Planner (基于优化的运动规划器)。
    2.  **$D_{dagger}$ (DAGGER数据)**：通过行为克隆(BC)训练初始策略，并进行交互增强。
        *   BC Loss: $L_{BC}(a^*, a_\theta) = L_{POSE}(a^*, a_\theta)$ (点匹配损失)
    3.  **$D_{ddpg}$ (RL探索数据)**：引入**Goal-Auxiliary**机制，为RL生成的轨迹寻找最近的“启发式目标”作为指导，解决缺乏专家示范的问题。
*   **总损失函数**：
    $$L_\theta = \lambda L_{BC}(a^*, a_\theta) + (1-\lambda)L_{DDPG}(s, a_\theta) + L_{AUX}(g, g_\theta)$$


### 核心定位：DDPG 是为了解决什么问题？

在 DDPG 之前，我们有两个主流阵营：
1.  **DQN (Deep Q-Network)**：非常擅长玩 Atari 游戏（离散动作：上下左右），它是 **Off-policy** 的（可以用 Replay Buffer 里的旧数据训练，效率高）。
    *   **死穴**：它没法做连续动作。因为 DQN 的核心是计算 $a^* = \arg\max_a Q(s, a)$。如果在连续空间（比如 $a$ 是 0.135 还是 0.136），要穷举所有 $a$ 算出 $Q$ 值再找最大值，计算量是无穷大的。
2.  **Policy Gradient (如 REINFORCE)**：直接输出动作概率，天生适合连续动作。
    *   **死穴**：它是 **On-policy** 的（边玩边学，旧数据扔掉），样本效率极低，机器人跑一天也学不会。

**DDPG 的诞生**：它结合了 **DQN 的高效率（Off-policy + Q函数）** 和 **Policy Gradient 的连续控制能力（Actor直接输出动作）**。

### 为什么 DDPG 是 Off-policy？

其背后的数学原理是 **贝尔曼最优方程 (Bellman Optimality Equation)**。

#### Q函数迭代公式

在 Q-Learning（以及 DDPG）中，我们追求的是“上帝视角”的真理——**最优 Q 值函数 ($Q^*$)**。它的定义是：在状态 $s$ 做动作 $a$，之后**永远都做最正确的选择**，能拿到的总分。

这个 $Q^*$ 满足一个不动点方程（Bellman Equation）：

$$ Q^*(s, a) = \mathbb{E}_{s'} [r + \gamma \max_{a'} Q^*(s', a')] $$

**这个公式的物理含义是**：
现在的价值 = 拿到手里的奖励 $r$ + $\gamma$ $\times$ (下一步最好动作的价值)。

#### 为什么叫“不动点”？为什么旧数据也能用？

这里的逻辑非常反直觉，请仔细看这个类比：

> **类比：迷宫寻宝**
> *   **目标**：计算从位置 A 到宝藏的距离（Q值）。
> *   **数据**：你有一堆以前的探险者留下的日记（Replay Buffer）。
> *   **情况**：有一个探险者是个傻子（旧的 Policy），他在位置 A 乱走到了位置 B，记录下距离是 10米。
> *   **问题**：这个傻子的数据对你有用吗？
> *   **答案**：**有用！**
>     因为“从 A 走到 B”是物理事实（环境动力学）。虽然他在 B 点之后可能乱走了，但如果你已经知道“从 B 点出发的最优路径（$\max Q^*(B, a')$）”是多少，你就可以反推出 A 点的价值：$Q(A) = 10 + \gamma \times Q(B)_{best}$。

**这就是 Off-policy 的本质：**
我们学习的是**环境的客观规律**（如果在 $s$ 做 $a$ 会得到 $r$ 并去 $s'$）。至于这个数据是谁跑出来的（是昨天的傻子策略，还是今天的聪明策略）并不重要。

只要我们不断用下式迭代，根据**压缩映射定理 (Contraction Mapping Theorem)**，Q 函数最终会收敛到唯一的**不动点** $Q^*$：

$$ Q_{new}(s, a) \leftarrow r + \gamma \max_{a'} Q_{old}(s', a') $$

---

### DDPG 的算法结构与设计思路

它是标准的 **Actor-Critic** 架构，但有两个特殊之处：**Deterministic (确定性)** 和 **Deep (深度网络)**。

#### 结构图解

DDPG 有四个核心网络（为了稳定训练）：

1.  **Actor 网络 ($\mu(s)$)**：
    *   **输入**：状态 $s$（如机器人关节角度、图像）。
    *   **输出**：确定的动作 $a$（如关节力矩）。
    *   **作用**：它就是为了解决 DQN 不能求 max 的问题。它直接告诉你：“别猜了，这个 $s$ 下，最大的 $Q$ 对应的动作就是这个 $a$”。

2.  **Critic 网络 ($Q(s, a)$)**：
    *   **输入**：状态 $s$ + 动作 $a$。
    *   **输出**：价值 $Q$。
    *   **作用**：裁判。它告诉 Actor：“你刚才选的动作 $a$，在 $s$ 下值多少分”。

3.  **Target Actor ($\mu'$) 和 Target Critic ($Q'$)**：
    *   这是深度学习训练的技巧，是原网络的“影子”，参数更新很慢，用来计算目标值（Target），防止训练发散。

#### 训练流程

**第一步：Critic 怎么学？（拟合贝尔曼方程）**
Critic 的任务是估算 Q 值。它使用 Replay Buffer 里的历史数据 $(s, a, r, s')$ 来更新。
目标值（Target $y$）的计算公式：

$$ y_i = r_i + \gamma \cdot Q'_{\phi'}(s'_i, \underbrace{\mu'_{\theta'}(s'_i)}_{\text{下一步的最优动作}}) $$

*   注意：这里没有像 DQN 那样写 $\max_{a'} Q(s', a')$，因为在连续空间算不出 max。
*   我们用 **Target Actor** $\mu'(s')$ 直接把下一步的最优动作算出来了，代进去就是 max。

Critic 的 Loss 就是让预测值逼近这个 Target：
$$ L_{critic} = \frac{1}{N} \sum (y_i - Q_{\phi}(s_i, a_i))^2 $$

**第二步：Actor 怎么学？（听 Critic 的指挥）**
Actor 的目的很简单：在状态 $s$ 下，输出一个动作 $a$，让 Critic 算出来的 $Q(s, a)$ 最大。
所以，我们把 Actor 的输出 $a = \mu(s)$ 塞给 Critic，然后对 Actor 的参数求梯度，让 $Q$ 变大。

梯度公式（链式法则）：
$$ \nabla_{\theta} J \approx \mathbb{E}_{s} [\nabla_a Q_{\phi}(s, a)|_{a=\mu(s)} \cdot \nabla_{\theta} \mu_{\theta}(s)] $$

*   物理含义：Actor 问 Critic：“我怎么改动作能让分变高？” Critic 说：“动作 $a$ 往左移一点分会变高”。Actor 就往左移一点。

---

#### 为什么课件里说 DDPG 是 "Deterministic"？

*   **传统 Policy Gradient (如 PPO)**：输出的是**高斯分布**的均值和方差 $\pi(a|s) \sim N(\mu, \sigma)$。采样时具有随机性。
*   **DDPG**：输出的是**确定的值** $a = \mu(s)$。

**既然是确定的，怎么探索环境（Exploration）呢？**
如果是确定的，机器人就会一直重复同样的动作，学不到新东西。所以 DDPG 在训练时，会**人为地在动作上加噪声**（如 Ornstein-Uhlenbeck 噪声或高斯噪声）：

$$ a_{real} = \mu(s) + \mathcal{N}(noise) $$

这就解释了为什么是 Off-policy：
*   **Behavior Policy (做动作的)**：$\mu(s) + \text{Noise}$
*   **Target Policy (我们要学的)**：$\mu(s)$
*   两者不同，所以是 Off-policy。

#### 为什么 DDPG 不适合 Locomotion 但适合 Grasping

##### 核心原因：Q 函数的“地形”不同

**Locomotion (行走)：悬崖峭壁上的走钢丝**
*   **任务特性**：足式机器人行走是一个 **欠驱动 (Underactuated)** 且 **动态不稳定** 的系统。
*   **Q 函数的敏感性**：
    *   在行走任务中，状态 $s$（机器人重心、关节速度）和动作 $a$（关节力矩）的配合必须极其精准。
    *   **敏感度极高**：在某个状态 $s$ 下，动作 $a$ 稍微偏离一点点（比如力矩大了 0.1Nm），机器人可能立马失去平衡摔倒。
    *   **Q 值突变**：这就导致 $Q(s, a)$ 函数在 $(s, a)$ 空间中是非常**陡峭 (Sharp)** 甚至**不连续**的。前一毫秒 $Q$ 值还很高（站着），动作稍微变一下，$Q$ 值瞬间跌零（摔了）。
*   **DDPG 的缺陷**：
    *   DDPG 学的是一个 **确定性策略 (Deterministic Policy, $\mu(s)$)**。它试图在这些“悬崖峭壁”上找到一条唯一的路。
    *   **过估计 (Overestimation)**：DDPG 有一个著名的毛病，就是容易**高估 Q 值**。在行走任务中，这种高估是致命的——它会错误地认为某个边缘动作（风险很高但还没摔）价值很高，结果一执行就摔了。
    *   一旦掉下悬崖（摔倒），梯度消失或变得非常杂乱，DDPG 很难爬回局部最优解。

**Grasping (抓取)：平原上的接近战**
*   **任务特性**：机械臂抓取（特别是 GA-DDPG 处理的 6D 抓取）通常是 **全驱动 (Fully Actuated)** 且 **准静态 (Quasi-static)** 的系统。
*   **Q 函数的平滑性**：
    *   **容错率高**：机械臂移动一点点，末端也就移动一点点。即使动作 $a$ 稍微不准，也就是抓偏了一点，不会导致系统“崩溃”（不像机器人摔倒那样不可挽回）。
    *   **Q 值渐变**：$Q(s, a)$ 函数通常比较**平滑 (Smooth)**。离物体越近，$Q$ 值越高；离得远，$Q$ 值越低。这里没有那么多“悬崖”。
*   **GA-DDPG 的适配性**：
    *   因为地形平滑，DDPG 的确定性策略更容易沿着梯度的方向慢慢滑向最高点（抓取点）。
    *   配合 **Goal-Auxiliary (目标辅助)**，实际上把 Q 函数塑造成了一个距离场（Distance Field），这进一步平滑了优化曲面，非常适合 DDPG 求解。

##### 为什么 Locomotion 更喜欢 PPO

既然 DDPG 不行，为什么 Locomotion 领域（如波士顿动力、Unitree Go2）都用 PPO？

*   **随机性策略 (Stochastic Policy)**：PPO 输出的是动作的分布（均值和方差）。
    *   在“走钢丝”时，PPO 不会只赌某一个特定的动作，而是保留了一定的随机性（方差）。这让它在面对不确定的动力学时更**鲁棒 (Robust)**。
*   **Trust Region (信任域)**：PPO 限制了每次更新的幅度。
    *   在 Locomotion 中，因为 Q 函数太陡峭，如果像 DDPG 那样步子迈大了，策略直接就崩了（Policy Collapse）。PPO 强行拉住更新步幅，保证“稳中求进”。

##### 为什么 Grasping 依然坚持用 DDPG 

既然 PPO 那么稳，为什么抓取任务（尤其是课件里的 GA-DDPG）还要用 DDPG？

**答案是：样本效率 (Sample Efficiency) 和 数据混合能力。**

*   **Locomotion 的训练环境**：通常在 Isaac Gym 等仿真器里跑。仿真器跑得飞快，几分钟能跑几亿步。**数据不值钱**。所以 PPO 这种 On-policy（跑一次扔一次，样本效率低）的算法也能接受，只要稳就行。
*   **Grasping 的痛点**：
    1.  **稀疏奖励**：很难抓到东西，如果是 On-policy 乱跑，可能跑一万年都抓不到一次，根本学不到东西。
    2.  **利用专家数据**：这正是 **GA-DDPG** 的核心！
        *   我们需要利用 **Expert (OMG Planner)** 的完美数据。
        *   我们需要利用 **Behavior Cloning (BC)** 的数据。
        *   **只有 Off-policy 算法 (如 DDPG/SAC)** 才能把别人的数据（Expert/BC）放进自己的 Replay Buffer 里反复学习。
        *   PPO 是 On-policy 的，它只能学“自己当前策略”跑出来的数据，没法直接利用专家数据（或者利用起来很麻烦）。


#### GA-DDPG 是什么？

**GA-DDPG (Goal-Auxiliary DDPG)**的背景是**6D 抓取**。原始 DDPG 很难训练，因为抓取是一个稀疏奖励（Sparse Reward）任务——要么抓到（1分），要么没抓到（0分）。机器人可能试了一万次都是0分，DDPG 就学废了。

**GA-DDPG 的改进点**：
它在 DDPG 的 Replay Buffer 中混合了三种数据：
1.  **$D_{expert}$**：从规划器（OMG Planner）算出来的完美轨迹。
2.  **$D_{dagger}$**：通过行为克隆（BC）初步学会的策略跑出来的数据。
3.  **$D_{ddpg}$**：DDPG 自己探索的数据。

**核心创新（Goal-Auxiliary）**：
对于 DDPG 自己瞎跑出来的数据（可能没抓到物体），GA-DDPG 不会只给它 0 分。它会用**事后诸葛亮（Hindsight）** 的思路：
*   虽然你没抓到目标 A，但你的手停在了 B 位置。
*   那我假装你的目标本来就是 B，那你这次操作不就满分了吗？
*   引入 $L_{AUX}(g, g_\theta)$ 辅助损失，帮助 Actor 理解几何空间关系。

**总结公式：**
$$ L_{total} = \lambda L_{BC} \text{(像专家学)} + (1-\lambda)L_{DDPG} \text{(自己探索)} + L_{AUX} \text{(理解几何目标)} $$
